# Parsing job listings from hh.ru using asynchronous programming

In [ ]:
import pandas as pd
from urllib.parse import urlparse, parse_qs

In [ ]:
import re
import nltk
import pymorphy3
from razdel import tokenize
from nltk.corpus import stopwords

nltk.download('stopwords')
russian_stopwords = set(stopwords.words('russian'))

morph = pymorphy3.MorphAnalyzer()

def clean_and_lemmatize_text(text, return_as_list=False):
    if not isinstance(text, str) or not text.strip():
        return [] if return_as_list else ""
    
    text = text.lower()
    
    text = re.sub(r'http\S+|www\.\S+', 'URL', text)
    text = re.sub(r'\S+@\S+', 'EMAIL', text)

    text = re.sub(r'\d+', 'NUM', text)
    
    tokens = [token.text for token in tokenize(text)]
    
    clean_tokens = [
        t for t in tokens 
        if re.search(r'[а-яёa-z]', t) or t == 'NUM'
    ]

    lemmatized_words = [morph.parse(word)[0].normal_form for word in clean_tokens]
    
    filtered_tokens = [word for word in lemmatized_words if word not in russian_stopwords]
    
    if return_as_list:
        return filtered_tokens
    
    return " ".join(filtered_tokens)


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/irinaaristova/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
import os
import re
import time
from multiprocessing.dummy import Pool as ThreadPool
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options


VACANCY_QUERY = "ML"
NUM_PAGES = 10
PROCESS_DELAY = 1  
MAX_PARALLEL_PROCESSES = 4
BASE_URL = "https://omsk.hh.ru/search/vacancy?search_field=name&search_field=company_name&search_field=description&text={}&enable_snippets=false&L_save_area=true&page={}"

def setup_driver():
    """Создаёт и настраивает экземпляр Chrome WebDriver."""
    chrome_options = Options()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--window-size=1920,1080")
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    
    prefs = {
        "profile.managed_default_content_settings.images": 2,
        "profile.default_content_setting_values.notifications": 2,
        "profile.managed_default_content_settings.stylesheets": 2
    }
    chrome_options.add_experimental_option("prefs", prefs)

    driver = webdriver.Chrome(options=chrome_options)
    return driver

In [ ]:
def parse_single_page(page_num, num_workers):
    """Парсит одну страницу поиска HeadHunter и извлекает подробное описание каждой вакансии."""
    initial_delay = page_num * PROCESS_DELAY % num_workers
    print(f"  [Поток {os.getpid()}] Задержка перед страницей {page_num}: {initial_delay:.2f} сек.")
    time.sleep(initial_delay)

    url = BASE_URL.format(VACANCY_QUERY, page_num)
    print(f"Парсинг страницы {page_num}... URL: {url}")
    all_vacancy_data = []
    driver = None

    try:
        driver = setup_driver()
        print(f"  [Поток {os.getpid()}] Открываем страницу поиска {page_num}...")
        driver.get(url)

        print(f"  [Поток {os.getpid()}] Ожидание загрузки страницы {page_num}...")
        time.sleep(3)

        print(f"  [Поток {os.getpid()}] Извлекаем HTML страницы {page_num}...")
        html_content = driver.page_source
        soup = BeautifulSoup(html_content, 'html.parser')

        title_elems = soup.find_all('a', {'data-qa': 'serp-item__title'})
        vacancy_items = []
        for t_elem in title_elems:
            card = t_elem.find_parent('div', class_=lambda c: c and 'vacancy-card' in c) or t_elem.find_parent('div')
            if card:
                vacancy_items.append((t_elem, card))

        if not vacancy_items:
            old_items = soup.find_all('div', {'data-qa': 'vacancy-serp__vacancy'})
            for item in old_items:
                t_elem = item.find('a', {'data-qa': 'serp-item__title'})
                if t_elem:
                    vacancy_items.append((t_elem, item))

        print(f"  [Поток {os.getpid()}] Найдено {len(vacancy_items)} вакансий на странице {page_num}")

        for title_elem, item in vacancy_items:
            title = title_elem.get_text(strip=True) if title_elem else "Не указано"

            employer_elem = (
                item.find('span', {'data-qa': 'vacancy-serp__vacancy-employer'}) or 
                item.find('a', {'data-qa': 'vacancy-serp__vacancy-employer'}) or 
                item.find('a', class_=lambda c: c and 'company' in c)
            )
            employer = employer_elem.get_text(strip=True) if employer_elem else "Не указано"

            salary_info = "Не указана"
            salary_min = None
            salary_max = None
            currency = None
            gross = None

            compensation_block = (
                item.find('span', {'data-qa': 'vacancy-serp__vacancy-compensation'}) or 
                item.find('span', class_=lambda x: x and 'compensation' in str(x)) or 
                item.find('div', class_=lambda x: x and 'compensation' in str(x))
            )
            if compensation_block:
                salary_text = compensation_block.get_text(strip=True)
                if salary_text:
                    salary_info = salary_text

                    numbers = re.findall(r'\d+(?:\s*\d+)*', salary_text)
                    nums = []
                    for num_str in numbers:
                        clean_num = int(re.sub(r'[\s\u00A0]', '', num_str))
                        nums.append(clean_num)

                    if len(nums) == 2:
                        salary_min, salary_max = nums[0], nums[1]
                        if salary_min > salary_max:
                            salary_min, salary_max = salary_max, salary_min
                    elif len(nums) == 1:
                        salary_lower = salary_text.lower()
                        if 'до' in salary_lower:
                            salary_max = nums[0]
                        elif 'от' in salary_lower:
                            salary_min = nums[0]
                        else:
                            salary_min = nums[0]

                    currency_match = re.search(r'(₽|USD|\$|EUR|€|KZT|₸|BYN|Br|UAH|₴|RUR|RUB|р\.|руб\.|рублей)', salary_text, re.IGNORECASE)
                    if currency_match:
                        raw_currency = currency_match.group(1).lower()
                        if raw_currency in ['₽', 'р.', 'руб.', 'рублей', 'rur', 'rub']:
                            currency = '₽'
                        elif raw_currency in ['usd', 'dollar', '$']:
                            currency = '$'
                        elif raw_currency in ['eur', 'euro', '€']:
                            currency = '€'
                        elif raw_currency in ['kzt', '₸']:
                            currency = '₸'

                    salary_lower = salary_text.lower()
                    if "до вычета налогов" in salary_lower or "до ндфл" in salary_lower or "до вычета" in salary_lower:
                        gross = True
                    elif "на руки" in salary_lower or "после вычета" in salary_lower or "на руки после" in salary_lower:
                        gross = False

            experience_text = "Не указан"
            experience_elem = (
                item.find('span', {'data-qa': lambda x: x and 'work-experience' in str(x)}) or 
                item.find('span', class_=lambda x: x and 'work-experience' in str(x))
            )
            if not experience_elem:
                experience_elem = item.find(string=re.compile(r'Опыт|Без опыта'))
            if experience_elem:
                experience_text = experience_elem.get_text(strip=True) if hasattr(experience_elem, 'get_text') else str(experience_elem).strip()

            city_elem = (
                item.find('span', {'data-qa': 'vacancy-serp__vacancy-address'}) or 
                item.find('div', class_=lambda x: x and 'address' in str(x)) or 
                item.find('span', class_=lambda x: x and 'address' in str(x))
            )
            city = city_elem.get_text(strip=True) if city_elem else "Не указано"

            vacancy_link = title_elem.get('href') if title_elem else None
            description_text = "Не указано"

            if vacancy_link:
                try:
                    driver.get(vacancy_link)
                    time.sleep(1) 
                    
                    vac_soup = BeautifulSoup(driver.page_source, 'html.parser')
                    desc_block = (
                        vac_soup.find('div', {'data-qa': 'vacancy-description'}) or 
                        vac_soup.find('div', class_=lambda c: c and 'g-user-content' in str(c))
                    )
                    
                    if desc_block:
                        description_text = desc_block.get_text(separator=' ', strip=True)
                except Exception as e_desc:
                    print(f"  [Поток {os.getpid()}] Ошибка при чтении описания ({vacancy_link}): {e_desc}")

            vacancy_data = {
                'Страница': page_num,
                'Вакансия': title,
                'Работодатель': employer,
                'Зарплата_сырая': salary_info,
                'Зарплата_мин': salary_min,
                'Зарплата_макс': salary_max,
                'Валюта': currency,
                'До_НДФЛ': gross,
                'Опыт_работы': experience_text,
                'Город': city,
                'Ссылка': vacancy_link,
                'Описание': description_text 
            }

            all_vacancy_data.append(vacancy_data)

        print(f"  [Поток {os.getpid()}] Обработано вакансий с описанием на странице {page_num}: {len(all_vacancy_data)}")

    except Exception as e:
        print(f"  Ошибка при парсинге страницы {page_num}: {e}")
        return []

    finally:
        if driver:
            print(f"  [Поток {os.getpid()}] Закрываем драйвер для страницы {page_num}")
            driver.quit()

    return all_vacancy_data


In [6]:
page_numbers = list(range(NUM_PAGES))
num_workers = min(NUM_PAGES, MAX_PARALLEL_PROCESSES)

print("--- Начало параллельного парсинга ---")
print(f"Запрос: {VACANCY_QUERY}, Страницы: 0 - {NUM_PAGES-1}")
print(f"Используется {num_workers} потоков для парсинга.")

tasks = [(p, num_workers) for p in page_numbers]
    
def worker_wrapper(args):
    return parse_single_page(args[0], args[1])

with ThreadPool(processes=num_workers) as pool:
    result_obj = pool.map_async(worker_wrapper, tasks)
    results = result_obj.get(timeout=600)

all_data_combined = []
for page_result in results:
    all_data_combined.extend(page_result)

print("--- Парсинг завершён ---")
print(f"Всего вакансий собрано: {len(all_data_combined)}")

if all_data_combined:
    df = pd.DataFrame(all_data_combined)
    csv_filename = f'vacancies_{VACANCY_QUERY}_pages_{NUM_PAGES}.csv'
    df.to_csv(csv_filename, index=False, encoding='utf-8-sig')
    print(f"Данные сохранены в файл: {csv_filename}")
else:
    print("Данные не были собраны.")

--- Начало параллельного парсинга ---
Запрос: ML, Страницы: 0 - 9
Используется 4 потоков для парсинга.
  [Поток 34943] Задержка перед страницей 0: 0.00 сек.
Парсинг страницы 0... URL: https://omsk.hh.ru/search/vacancy?search_field=name&search_field=company_name&search_field=description&text=ML&enable_snippets=false&L_save_area=true&page=0
  [Поток 34943] Задержка перед страницей 2: 2.00 сек.
  [Поток 34943] Задержка перед страницей 3: 3.00 сек.
  [Поток 34943] Задержка перед страницей 1: 1.00 сек.
  [Поток 34943] Открываем страницу поиска 0...
Парсинг страницы 1... URL: https://omsk.hh.ru/search/vacancy?search_field=name&search_field=company_name&search_field=description&text=ML&enable_snippets=false&L_save_area=true&page=1
  [Поток 34943] Открываем страницу поиска 1...
Парсинг страницы 2... URL: https://omsk.hh.ru/search/vacancy?search_field=name&search_field=company_name&search_field=description&text=ML&enable_snippets=false&L_save_area=true&page=2
  [Поток 34943] Открываем страницу 

In [ ]:
import pandas as pd

df = pd.read_csv('vacancies_ML_pages_10.csv')

df['Описание_cleaned'] = df['Описание'].apply(clean_and_lemmatize_text)

df.to_csv('vacancies_ML_processed.csv', index=False, encoding='utf-8-sig')
